In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor()])
full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
full_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_set = Subset(full_train, range(10000))
test_set = Subset(full_test, range(2000))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=2000, shuffle=False)

test_images, test_labels = next(iter(test_loader))
test_images_np = test_images.numpy().squeeze()

def calculate_metrics(clean_imgs, recon_imgs):
    clean_np = clean_imgs.detach().cpu().numpy().squeeze()
    recon_np = recon_imgs.detach().cpu().numpy().squeeze()

    mse = np.mean((clean_np - recon_np) ** 2)
    mae = np.mean(np.abs(clean_np - recon_np))

    ssim_vals = []
    for i in range(len(clean_np)):
        val = ssim(clean_np[i], recon_np[i], data_range=1.0)
        ssim_vals.append(val)
    mean_ssim = np.mean(ssim_vals)

    return mse, mae, mean_ssim

def add_gaussian_noise(imgs, sigma):
    noisy = imgs + torch.randn_like(imgs) * sigma
    return torch.clamp(noisy, 0.0, 1.0)

def add_salt_and_pepper_noise(imgs, p):
    noisy = imgs.clone()
    num_noise = int(p * 28 * 28)
    for i in range(len(noisy)):
        coords_i = np.random.randint(0, 28, num_noise)
        coords_j = np.random.randint(0, 28, num_noise)
        half = num_noise // 2
        noisy[i, 0, coords_i[:half], coords_j[:half]] = 0.0
        noisy[i, 0, coords_i[half:], coords_j[half:]] = 1.0
    return noisy

class FullyConnectedAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28))
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

class ConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, padding=0),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, padding=0),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

class ConvVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)

        self.decoder_fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.decoder_fc(z))
        return self.decoder_conv(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_model(model, train_loader, test_images, epochs=20, lr=1e-3, noise_type=None, noise_level=0.0):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss(reduction='sum')

    train_losses = []
    val_losses = []

    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            if noise_type == 'gaussian':
                inputs = add_gaussian_noise(imgs, noise_level)
            elif noise_type == 'salt_pepper':
                inputs = add_salt_and_pepper_noise(imgs, noise_level)
            else:
                inputs = imgs

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, imgs)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        with torch.no_grad():
            t_imgs = test_images.to(device)
            if noise_type == 'gaussian':
                t_inputs = add_gaussian_noise(t_imgs, noise_level)
            elif noise_type == 'salt_pepper':
                t_inputs = add_salt_and_pepper_noise(t_imgs, noise_level)
            else:
                t_inputs = t_imgs
            t_outputs = model(t_inputs)
            val_loss = criterion(t_outputs, t_imgs).item() / len(t_imgs)
            val_losses.append(val_loss)

    elapsed = time.time() - start_time
    return train_losses, val_losses, elapsed

print("=== Training Fully Connected Autoencoder ===")
fc_ae = FullyConnectedAE(latent_dim=16)
fc_train_losses, fc_val_losses, fc_time = train_model(fc_ae, train_loader, test_images, epochs=20)

fc_ae.eval()
with torch.no_grad():
    fc_recon = fc_ae(test_images.to(device)).cpu()
fc_mse, fc_mae, fc_ssim = calculate_metrics(test_images, fc_recon)

plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(2, 10, i + 1)
    plt.imshow(test_images[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("Orig", fontsize=8)
    plt.subplot(2, 10, i + 11)
    plt.imshow(fc_recon[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("FC Recon", fontsize=8)
plt.tight_layout()
plt.savefig("plot1_fc_reconstruction.png")
plt.close()

plt.figure(figsize=(6, 4))
plt.plot(range(1, 21), fc_train_losses, label='Train Loss')
plt.plot(range(1, 21), fc_val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('FC-AE Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig("plot2_fc_loss.png")
plt.close()

print("=== Training Convolutional Autoencoder ===")
cae = ConvAE()
cae_train_losses, cae_val_losses, cae_time = train_model(cae, train_loader, test_images, epochs=20)

cae.eval()
with torch.no_grad():
    cae_recon = cae(test_images.to(device)).cpu()
cae_mse, cae_mae, cae_ssim = calculate_metrics(test_images, cae_recon)

plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(test_images[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("Orig", fontsize=8)
    plt.subplot(3, 10, i + 11)
    plt.imshow(fc_recon[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("FC-AE", fontsize=8)
    plt.subplot(3, 10, i + 21)
    plt.imshow(cae_recon[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("CAE", fontsize=8)
plt.tight_layout()
plt.savefig("plot3_ae_vs_cae.png")
plt.close()

print("=== Training Denoising Autoencoder (Gaussian sigma=0.2) ===")
dae = ConvAE()
dae_train_losses, dae_val_losses, dae_time = train_model(dae, train_loader, test_images, epochs=20, noise_type='gaussian', noise_level=0.2)

dae.eval()
noisy_test_g2 = add_gaussian_noise(test_images, 0.2)
with torch.no_grad():
    dae_recon = dae(noisy_test_g2.to(device)).cpu()
dae_mse, dae_mae, dae_ssim = calculate_metrics(test_images, dae_recon)

plt.figure(figsize=(10, 3))
for i in range(10):
    plt.subplot(3, 10, i + 1)
    plt.imshow(test_images[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("Clean", fontsize=8)
    plt.subplot(3, 10, i + 11)
    plt.imshow(noisy_test_g2[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("Noisy", fontsize=8)
    plt.subplot(3, 10, i + 21)
    plt.imshow(dae_recon[i].squeeze(), cmap='gray')
    plt.axis('off')
    if i == 0: plt.title("Denoised", fontsize=8)
plt.tight_layout()
plt.savefig("plot4_denoising_samples.png")
plt.close()

print("=== Denoising Performance Across Noise Levels ===")
noise_results = []
gauss_sigmas = [0.1, 0.2, 0.3]
sp_levels = [0.05, 0.10, 0.20]

for sigma in gauss_sigmas:
    m = ConvAE()
    train_model(m, train_loader, test_images, epochs=20, noise_type='gaussian', noise_level=sigma)
    m.eval()
    with torch.no_grad():
        n_img = add_gaussian_noise(test_images, sigma)
        rec = m(n_img.to(device)).cpu()
    mse, mae, ssim_v = calculate_metrics(test_images, rec)
    noise_results.append({'type': 'Gaussian', 'level': sigma, 'mse': mse, 'mae': mae, 'ssim': ssim_v})

for p in sp_levels:
    m = ConvAE()
    train_model(m, train_loader, test_images, epochs=20, noise_type='salt_pepper', noise_level=p)
    m.eval()
    with torch.no_grad():
        n_img = add_salt_and_pepper_noise(test_images, p)
        rec = m(n_img.to(device)).cpu()
    mse, mae, ssim_v = calculate_metrics(test_images, rec)
    noise_results.append({'type': 'Salt & Pepper', 'level': p, 'mse': mse, 'mae': mae, 'ssim': ssim_v})

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
g_levels = [r['level'] for r in noise_results if r['type'] == 'Gaussian']
g_mse = [r['mse'] for r in noise_results if r['type'] == 'Gaussian']
g_mae = [r['mae'] for r in noise_results if r['type'] == 'Gaussian']
g_ssim = [r['ssim'] for r in noise_results if r['type'] == 'Gaussian']

sp_lvls = [r['level'] for r in noise_results if r['type'] == 'Salt & Pepper']
sp_mse = [r['mse'] for r in noise_results if r['type'] == 'Salt & Pepper']
sp_mae = [r['mae'] for r in noise_results if r['type'] == 'Salt & Pepper']
sp_ssim = [r['ssim'] for r in noise_results if r['type'] == 'Salt & Pepper']

axes[0].plot(g_levels, g_mse, 'o-', label='Gaussian')
axes[0].plot(sp_lvls, sp_mse, 's--', label='Salt & Pepper')
axes[0].set_title('MSE vs Noise')
axes[0].legend()

axes[1].plot(g_levels, g_mae, 'o-', label='Gaussian')
axes[1].plot(sp_lvls, sp_mae, 's--', label='Salt & Pepper')
axes[1].set_title('MAE vs Noise')
axes[1].legend()

axes[2].plot(g_levels, g_ssim, 'o-', label='Gaussian')
axes[2].plot(sp_lvls, sp_ssim, 's--', label='Salt & Pepper')
axes[2].set_title('SSIM vs Noise')
axes[2].legend()

plt.tight_layout()
plt.savefig("plot5_noise_vs_quality.png")
plt.close()

print("=== Training Variational Autoencoder (VAE) ===")
vae = ConvVAE(latent_dim=2).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)

def vae_loss_function(recon_x, x, mu, logvar):
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE, KLD

vae_train_losses = []
vae_val_losses = []
vae_start_time = time.time()

for epoch in range(20):
    vae.train()
    train_loss = 0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)
        vae_optimizer.zero_grad()
        recon, mu, logvar = vae(imgs)
        bce, kld = vae_loss_function(recon, imgs, mu, logvar)
        loss = bce + kld
        loss.backward()
        vae_optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader.dataset)
    vae_train_losses.append(train_loss)

    vae.eval()
    with torch.no_grad():
        t_imgs = test_images.to(device)
        t_recon, t_mu, t_logvar = vae(t_imgs)
        t_bce, t_kld = vae_loss_function(t_recon, t_imgs, t_mu, t_logvar)
        val_loss = (t_bce + t_kld).item() / len(t_imgs)
        vae_val_losses.append(val_loss)

vae_time = time.time() - vae_start_time

vae.eval()
with torch.no_grad():
    v_recon, v_mu, v_logvar = vae(test_images.to(device))
    v_bce, v_kld = vae_loss_function(v_recon, test_images.to(device), v_mu, v_logvar)

vae_rec_loss = (v_bce / len(test_images)).item()
vae_kl_loss = (v_kld / len(test_images)).item()
vae_total_loss = vae_rec_loss + vae_kl_loss
vae_mse, vae_mae, vae_ssim = calculate_metrics(test_images, v_recon.cpu())

plt.figure(figsize=(7, 6))
scatter = plt.scatter(v_mu[:, 0].cpu().numpy(), v_mu[:, 1].cpu().numpy(), c=test_labels.numpy(), cmap='tab10', alpha=0.7, s=10)
plt.colorbar(scatter)
plt.title('VAE 2D Latent Space Visualization')
plt.xlabel('z1')
plt.ylabel('z2')
plt.tight_layout()
plt.savefig("plot6_vae_latent_space.png")
plt.close()

plt.figure(figsize=(5, 5))
with torch.no_grad():
    z_samples = torch.randn(25, 2).to(device)
    gen_imgs = vae.decode(z_samples).cpu()
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(gen_imgs[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.savefig("plot7_vae_generated.png")
plt.close()

z_A = v_mu[0:1]
z_B = v_mu[1:2]
alphas = np.linspace(0, 1, 11)
plt.figure(figsize=(11, 1.5))
with torch.no_grad():
    for idx, a in enumerate(alphas):
        z_interp = (1 - a) * z_A + a * z_B
        interp_img = vae.decode(z_interp).cpu().squeeze()
        plt.subplot(1, 11, idx + 1)
        plt.imshow(interp_img, cmap='gray')
        plt.title(f"{a:.1f}", fontsize=8)
        plt.axis('off')
plt.tight_layout()
plt.savefig("plot8_vae_interpolation.png")
plt.close()

plt.figure(figsize=(6, 4))
plt.plot(range(1, 21), vae_train_losses, label='Train Loss')
plt.plot(range(1, 21), vae_val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Total Loss (BCE + KLD)')
plt.title('VAE Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig("plot9_vae_loss.png")
plt.close()

fc_errors = torch.mean((test_images - fc_recon) ** 2, dim=[1, 2, 3]).numpy()
cae_errors = torch.mean((test_images - cae_recon) ** 2, dim=[1, 2, 3]).numpy()
dae_errors = torch.mean((test_images - dae_recon) ** 2, dim=[1, 2, 3]).numpy()
vae_errors = torch.mean((test_images - v_recon.cpu()) ** 2, dim=[1, 2, 3]).numpy()

plt.figure(figsize=(8, 4.5))
plt.hist(fc_errors, bins=30, alpha=0.5, label='FC-AE')
plt.hist(cae_errors, bins=30, alpha=0.5, label='CAE')
plt.hist(dae_errors, bins=30, alpha=0.5, label='DAE')
plt.hist(vae_errors, bins=30, alpha=0.5, label='VAE')
plt.xlabel('Per-Image Reconstruction Error (MSE)')
plt.ylabel('Frequency')
plt.title('Reconstruction Error Distribution Across Architectures')
plt.legend()
plt.tight_layout()
plt.savefig("plot10_error_distribution.png")
plt.close()

top5_idx = np.argsort(cae_errors)[-5:][::-1]
plt.figure(figsize=(10, 2.5))
for i, idx in enumerate(top5_idx):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[idx].squeeze(), cmap='gray')
    plt.title(f"Orig #{idx}", fontsize=8)
    plt.axis('off')

    plt.subplot(2, 5, i + 6)
    plt.imshow(cae_recon[idx].squeeze(), cmap='gray')
    plt.title(f"MSE: {cae_errors[idx]:.4f}", fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.savefig("plot11_high_error_samples.png")
plt.close()

print("=== Latent Dimension Study for Basic FC-AE ===")
dims = [2, 8, 16, 32]
dim_results = []

for d in dims:
    m = FullyConnectedAE(latent_dim=d)
    train_model(m, train_loader, test_images, epochs=20)
    m.eval()
    with torch.no_grad():
        rec = m(test_images.to(device)).cpu()
    mse, mae, ssim_v = calculate_metrics(test_images, rec)
    dim_results.append({'dim': d, 'mse': mse, 'ssim': ssim_v, 'params': count_params(m)})

plt.figure(figsize=(6, 4))
plt.plot(dims, [r['mse'] for r in dim_results], 'o-')
plt.xlabel('Latent Dimension (d_z)')
plt.ylabel('Reconstruction MSE')
plt.title('Latent Dimension vs Reconstruction MSE')
plt.grid(True)
plt.tight_layout()
plt.savefig("plot12_latent_dim_vs_mse.png")
plt.close()

print("\n" + "="*60)
print("SECTION 9: Basic FC Autoencoder Metrics")
print("="*60)
print(f"Test MSE : {fc_mse:.6f}")
print(f"Test MAE : {fc_mae:.6f}")
print(f"Mean SSIM: {fc_ssim:.6f}")

print("\n" + "="*60)
print("SECTION 11: FC-AE vs CAE Comparison")
print("="*60)
print(f"{'Model':<20} | {'MSE':<10} | {'MAE':<10} | {'SSIM':<10} | {'Parameters':<10}")
print("-" * 65)
print(f"{'Fully Connected AE':<20} | {fc_mse:<10.6f} | {fc_mae:<10.6f} | {fc_ssim:<10.6f} | {count_params(fc_ae):<10}")
print(f"{'Convolutional AE':<20} | {cae_mse:<10.6f} | {cae_mae:<10.6f} | {cae_ssim:<10.6f} | {count_params(cae):<10}")

print("\n" + "="*60)
print("SECTION 14: Noise Level vs Reconstruction Quality Table")
print("="*60)
print(f"{'Noise Type':<18} | {'Level':<6} | {'MSE':<10} | {'MAE':<10} | {'SSIM':<10}")
print("-" * 65)
for r in noise_results:
    print(f"{r['type']:<18} | {r['level']:<6.2f} | {r['mse']:<10.6f} | {r['mae']:<10.6f} | {r['ssim']:<10.6f}")

print("\n" + "="*60)
print("SECTION 21: VAE Specific Metrics")
print("="*60)
print(f"Reconstruction Loss : {vae_rec_loss:.6f}")
print(f"KL Loss             : {vae_kl_loss:.6f}")
print(f"Total Loss          : {vae_total_loss:.6f}")
print(f"Test MSE            : {vae_mse:.6f}")
print(f"Test MAE            : {vae_mae:.6f}")
print(f"Mean SSIM           : {vae_ssim:.6f}")

print("\n" + "="*60)
print("SECTION 25: Consolidated Results Table")
print("="*60)
print(f"{'Model':<18} | {'MSE':<9} | {'MAE':<9} | {'SSIM':<9} | {'Parameters':<10} | {'Time (s)':<8}")
print("-" * 75)
print(f"{'FC Autoencoder':<18} | {fc_mse:<9.6f} | {fc_mae:<9.6f} | {fc_ssim:<9.6f} | {count_params(fc_ae):<10} | {fc_time:<8.2f}")
print(f"{'Conv. Autoencoder':<18} | {cae_mse:<9.6f} | {cae_mae:<9.6f} | {cae_ssim:<9.6f} | {count_params(cae):<10} | {cae_time:<8.2f}")
print(f"{'Denoising CAE':<18} | {dae_mse:<9.6f} | {dae_mae:<9.6f} | {dae_ssim:<9.6f} | {count_params(dae):<10} | {dae_time:<8.2f}")
print(f"{'VAE':<18} | {vae_mse:<9.6f} | {vae_mae:<9.6f} | {vae_ssim:<9.6f} | {count_params(vae):<10} | {vae_time:<8.2f}")

print("\n" + "="*60)
print("SECTION 27: Additional Latent-Dimension Study Table")
print("="*60)
print(f"{'Latent Dimension':<18} | {'MSE':<10} | {'SSIM':<10} | {'Parameters':<10}")
print("-" * 55)
for r in dim_results:
    print(f"{r['dim']:<18} | {r['mse']:<10.6f} | {r['ssim']:<10.6f} | {r['params']:<10}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 461kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.02MB/s]


=== Training Fully Connected Autoencoder ===
=== Training Convolutional Autoencoder ===
=== Training Denoising Autoencoder (Gaussian sigma=0.2) ===
=== Denoising Performance Across Noise Levels ===
=== Training Variational Autoencoder (VAE) ===
=== Latent Dimension Study for Basic FC-AE ===

SECTION 9: Basic FC Autoencoder Metrics
Test MSE : 0.026545
Test MAE : 0.067755
Mean SSIM: 0.680747

SECTION 11: FC-AE vs CAE Comparison
Model                | MSE        | MAE        | SSIM       | Parameters
-----------------------------------------------------------------
Fully Connected AE   | 0.026545   | 0.067755   | 0.680747   | 211040    
Convolutional AE     | 0.003595   | 0.018320   | 0.956844   | 74497     

SECTION 14: Noise Level vs Reconstruction Quality Table
Noise Type         | Level  | MSE        | MAE        | SSIM      
-----------------------------------------------------------------
Gaussian           | 0.10   | 0.003409   | 0.017818   | 0.962782  
Gaussian           | 0.20   

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor()])
full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
full_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_set = Subset(full_train, range(10000))
test_set = Subset(full_test, range(2000))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=2000, shuffle=False)

test_images, test_labels = next(iter(test_loader))

def calculate_metrics(clean_imgs, recon_imgs):
    clean_np = clean_imgs.detach().cpu().numpy().squeeze()
    recon_np = recon_imgs.detach().cpu().numpy().squeeze()
    mse = np.mean((clean_np - recon_np) ** 2)
    mae = np.mean(np.abs(clean_np - recon_np))
    ssim_vals = [ssim(clean_np[i], recon_np[i], data_range=1.0) for i in range(len(clean_np))]
    return mse, mae, np.mean(ssim_vals)

def add_gaussian_noise(imgs, sigma):
    noisy = imgs + torch.randn_like(imgs) * sigma
    return torch.clamp(noisy, 0.0, 1.0)

def add_salt_and_pepper_noise(imgs, p):
    noisy = imgs.clone()
    num_noise = int(p * 28 * 28)
    for i in range(len(noisy)):
        coords_i = np.random.randint(0, 28, num_noise)
        coords_j = np.random.randint(0, 28, num_noise)
        half = num_noise // 2
        noisy[i, 0, coords_i[:half], coords_j[:half]] = 0.0
        noisy[i, 0, coords_i[half:], coords_j[half:]] = 1.0
    return noisy

# --- Exercise 1: Flexible Latent Dimension ConvAE ---
class FlexibleConvAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, padding=0),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, padding=0),
            nn.Flatten()
        )
        self.fc_enc = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.Unflatten(1, (64, 7, 7)),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.encoder_conv(x)
        z = self.fc_enc(h)
        h_dec = torch.relu(self.fc_dec(z))
        return self.decoder_conv(h_dec)

# --- Exercise 4: Transposed Conv Autoencoder ---
class TransposedConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

# --- Exercise 5 & 6: Conv VAE with Beta parameter ---
class FlexibleVAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(64 * 7 * 7, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.decoder_conv = nn.Sequential(
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder_conv(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.decoder_fc(z))
        return self.decoder_conv(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def train_ae(model, train_loader, test_images, epochs=15, lr=1e-3, noise_type=None, noise_level=0.0):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss(reduction='sum')
    for epoch in range(epochs):
        model.train()
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            inputs = add_gaussian_noise(imgs, noise_level) if noise_type == 'gaussian' else (add_salt_and_pepper_noise(imgs, noise_level) if noise_type == 'salt_pepper' else imgs)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, imgs)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        t_imgs = test_images.to(device)
        t_inputs = add_gaussian_noise(t_imgs, noise_level) if noise_type == 'gaussian' else (add_salt_and_pepper_noise(t_imgs, noise_level) if noise_type == 'salt_pepper' else t_imgs)
        recon = model(t_inputs).cpu()
    return calculate_metrics(test_images, recon)

# --- Ex 1 Execution ---
print("Ex 1: ConvAE Latent Dims Study")
ex1_results = {}
for d in [4, 8, 16, 32]:
    m = FlexibleConvAE(latent_dim=d)
    mse, mae, ssim_val = train_ae(m, train_loader, test_images, epochs=15)
    ex1_results[d] = (mse, ssim_val)
    print(f"Latent Dim {d} | MSE: {mse:.6f} | SSIM: {ssim_val:.6f}")

# --- Ex 3 Execution ---
print("\nEx 3: DAE Noise Level Comparison")
dae_base = FlexibleConvAE(latent_dim=16)
mse1, mae1, ssim1 = train_ae(dae_base, train_loader, test_images, epochs=15, noise_type='gaussian', noise_level=0.1)
mse2, mae2, ssim2 = train_ae(dae_base, train_loader, test_images, epochs=15, noise_type='gaussian', noise_level=0.3)
print(f"Gaussian Noise 0.1 | SSIM: {ssim1:.6f}")
print(f"Gaussian Noise 0.3 | SSIM: {ssim2:.6f}")

# --- Ex 4 Execution ---
print("\nEx 4: Transposed Conv AE")
trans_ae = TransposedConvAE()
t_mse, t_mae, t_ssim = train_ae(trans_ae, train_loader, test_images, epochs=15)
print(f"Transposed Conv AE | MSE: {t_mse:.6f} | SSIM: {t_ssim:.6f}")

# --- Corrected VAE Training Helper ---
def train_vae(model, train_loader, epochs=15, lr=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for imgs, _ in train_loader:
            imgs = imgs.to(device)
            optimizer.zero_grad()
            recon, mu, logvar = model(imgs)

            # Reconstruction loss + KL divergence
            bce = nn.functional.binary_cross_entropy(recon, imgs, reduction='sum')
            kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = bce + kld

            loss.backward()
            optimizer.step()

# --- Ex 7 Plot: 8x8 Grid Generation ---
vae2 = FlexibleVAE(latent_dim=2).to(device)
train_vae(vae2, train_loader, epochs=15)  # <--- Using train_vae instead of train_ae

plt.figure(figsize=(8, 8))
with torch.no_grad():
    z_samples = torch.randn(64, 2).to(device)
    gen_imgs = vae2.decode(z_samples).cpu()

for i in range(64):
    plt.subplot(8, 8, i + 1)
    plt.imshow(gen_imgs[i].squeeze(), cmap='gray')
    plt.axis('off')
plt.tight_layout()
plt.savefig("ex7_large_vae_grid.png")
plt.close()

# --- Ex 8 Plot: Specific Region Interpolation ---
vae2.eval()
with torch.no_grad():
    _, mu_all, _ = vae2(test_images.to(device))

labels_np = test_labels.numpy()
idx_digit1 = np.where(labels_np == 1)[0][0]
idx_digit7 = np.where(labels_np == 7)[0][0]

z_1 = mu_all[idx_digit1:idx_digit1+1]
z_7 = mu_all[idx_digit7:idx_digit7+1]

alphas = np.linspace(0, 1, 10)
plt.figure(figsize=(10, 1.5))
with torch.no_grad():
    for idx, a in enumerate(alphas):
        z_interp = (1 - a) * z_1 + a * z_7
        img = vae2.decode(z_interp).cpu().squeeze()
        plt.subplot(1, 10, idx + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{a:.1f}", fontsize=8)
        plt.axis('off')
plt.tight_layout()
plt.savefig("ex8_digit_region_interpolation.png")
plt.close()

Ex 1: ConvAE Latent Dims Study
Latent Dim 4 | MSE: 0.037183 | SSIM: 0.583494
Latent Dim 8 | MSE: 0.023587 | SSIM: 0.744024
Latent Dim 16 | MSE: 0.013026 | SSIM: 0.858932
Latent Dim 32 | MSE: 0.007412 | SSIM: 0.920035

Ex 3: DAE Noise Level Comparison
Gaussian Noise 0.1 | SSIM: 0.846801
Gaussian Noise 0.3 | SSIM: 0.845021

Ex 4: Transposed Conv AE
Transposed Conv AE | MSE: 0.000779 | SSIM: 0.988625
